# Example Notebook — ASR Robustness Evaluation (Step-by-Step)

## Feel free to create your own notebook, this is just for you to get it started
**Goal:** Evaluate **≥ 5** state-of-the-art ASR models on your own recordings under:
- **Clean** (quiet room)
- **Synthetic additive noise** (2 noise types × 2 SNR levels)
- **Real-world recordings** (2 environments)
- *(optional) Reverberation* (1–2 simulated rooms)
- *(Bonus)* **multilingual/code-mixed speech  or special-population (children, older adults, disorders) evaluation** 

You will compute:
- **WER**, (optional **CER**)
- **Substitutions / Insertions / Deletions** (S/I/D)
- A focused analysis comparing **7 grammatical** vs **3ungrammatical** sentences

> **No model training.** Only inference + evaluation + analysis.


## 0) What you need before running this notebook

### A. Create your 10 sentences
- **7 grammatically correct**
- **3 grammatically incorrect** (intentionally incorrect)

### Optional format: 
Save transcripts into: `project/sentences/ground_truth.csv` with columns:
- `sentence_id` (s01 … s10)
- `text` (the ground truth)
- `is_grammatical` (1 for grammatical, 0 for ungrammatical)

#### Record audio (your own voice)
Record and Save into:

- `project/audio/clean/s01.wav ... s10.wav`
- `project/audio/real/env1/s01.wav ... s10.wav`
- `project/audio/real/env2/s01.wav ... s10.wav`

**Real-world environment examples (choose any):**
- walking outdoors on a street 
- cafeteria / restaurant ambience
- inside a moving car / bus
- cooking at home (fan + utensils)
- quiet whisper/soft speech scenario 

✅ First run models on **as-recorded** real-world audio (no enhancement yet). Enhancement is Notebook 02.


## 1) Install dependencies (run once)

If you use a fresh environment (e.g., Colab):

```bash
pip install -U transformers accelerate torchaudio soundfile librosa jiwer pandas numpy scipy matplotlib
pip install -U openai-whisper
```

Notes:
- Whisper requires `ffmpeg` (often preinstalled on Colab; otherwise install system ffmpeg).
- Some HF models may require GPU for reasonable speed.


In [ ]:
import os, re, math, json
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
import matplotlib.pyplot as plt

from jiwer import compute_measures, cer
from scipy.signal import butter, sosfilt
from scipy.signal import fftconvolve

plt.rcParams["figure.figsize"] = (10, 4)


## 2) Configure paths (edit PROJECT_ROOT if needed)


In [ ]:
PROJECT_ROOT = "./project"

GT_PATH = os.path.join(PROJECT_ROOT, "sentences", "ground_truth.csv")

CLEAN_DIR = os.path.join(PROJECT_ROOT, "audio", "clean")
REAL_DIR  = os.path.join(PROJECT_ROOT, "audio", "real")        # contains env1/, env2/
SYN_DIR   = os.path.join(PROJECT_ROOT, "audio", "synthetic")   # will be created

OUT_DIR   = os.path.join(PROJECT_ROOT, "outputs")
MET_DIR   = os.path.join(OUT_DIR, "metrics")
FIG_DIR   = os.path.join(OUT_DIR, "figures")

for d in [SYN_DIR, OUT_DIR, MET_DIR, FIG_DIR]:
    os.makedirs(d, exist_ok=True)

SR = 16000


## 3) Load ground truth and validate formatting


In [ ]:
gt = pd.read_csv(GT_PATH)
required_cols = {"sentence_id","text","is_grammatical"}
missing = required_cols - set(gt.columns)
if missing:
    raise ValueError(f"ground_truth.csv missing columns: {missing}")

gt["sentence_id"] = gt["sentence_id"].astype(str)
gt["is_grammatical"] = gt["is_grammatical"].astype(int)

gt.sort_values("sentence_id").head(10)


## 4) Audio helpers + sanity checks

This section checks:
- all s01..s10 exist in clean
- real env folders contain the same sentence ids


In [ ]:
def list_wavs(folder):
    if not os.path.exists(folder):
        return []
    return sorted([f for f in os.listdir(folder) if f.lower().endswith(".wav")])

def expect_sentence_ids(files):
    return sorted([os.path.splitext(f)[0] for f in files])

clean_files = list_wavs(CLEAN_DIR)
print("Clean wav count:", len(clean_files))
print("Clean IDs:", expect_sentence_ids(clean_files))

# Check real envs
if os.path.exists(REAL_DIR):
    envs = sorted([d for d in os.listdir(REAL_DIR) if os.path.isdir(os.path.join(REAL_DIR,d))])
else:
    envs = []
print("Real environments found:", envs)

for env in envs:
    files = list_wavs(os.path.join(REAL_DIR, env))
    print(env, "count:", len(files), "IDs:", expect_sentence_ids(files)[:3], "...")

# Quick consistency check
gt_ids = sorted(gt["sentence_id"].tolist())
clean_ids = expect_sentence_ids(clean_files)
if gt_ids != clean_ids:
    print("WARNING: ground_truth sentence_ids and clean wav IDs do not match.")


## 5) Synthetic additive noise generation (2 noise types × 2 SNR levels)

**You must use 2 noise types** (default: white + pink/traffic placeholder).

Options:
- **White noise** (generated)
- **Pink noise** (generated, proxy for “realistic” low-frequency noise)
- **Traffic noise file**: you may provide `project/noises/traffic.wav` (recommended)

SNR levels:
- **20 dB** (mild noise)
- **0 dB** (moderate noise)

Output folders created:
- `project/audio/synthetic/noise_white_snr20/`
- `project/audio/synthetic/noise_white_snr10/`
- `project/audio/synthetic/noise_traffic_snr20/` ... etc


In [ ]:
NOISE_DIR = os.path.join(PROJECT_ROOT, "noises")
TRAFFIC_WAV = os.path.join(NOISE_DIR, "traffic.wav")  # optional file you can add

SNR_LEVELS = [20, 10]
NOISE_TYPES = ["white", "traffic"]  # keep 2 types; you can replace traffic with pink if you want

def load_wav(path, sr=SR):
    y, _ = librosa.load(path, sr=sr, mono=True)
    return y.astype(np.float32)

def save_wav(path, y, sr=SR):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    sf.write(path, y, sr)

def rms(x):
    return float(np.sqrt(np.mean(x**2) + 1e-12))

def match_len(x, n):
    if len(x) == n:
        return x
    if len(x) > n:
        return x[:n]
    return np.pad(x, (0, n-len(x)))

def white_noise(n):
    x = np.random.randn(n).astype(np.float32)
    x = x / (rms(x) + 1e-12)
    return x

def pink_noise(n):
    # simple 1/f noise using FFT shaping
    x = np.random.randn(n).astype(np.float32)
    X = np.fft.rfft(x)
    freqs = np.linspace(1, len(X), len(X))
    X = X / freqs
    y = np.fft.irfft(X, n=n).astype(np.float32)
    y = y / (rms(y) + 1e-12)
    return y

def add_noise_snr(clean, noise, snr_db):
    noise = match_len(noise, len(clean))
    clean_p = np.mean(clean**2) + 1e-12
    noise_p = np.mean(noise**2) + 1e-12
    desired_noise_p = clean_p / (10**(snr_db/10))
    scale = math.sqrt(desired_noise_p / noise_p)
    return (clean + scale*noise).astype(np.float32)

def get_noise(noise_type, n):
    if noise_type == "white":
        return white_noise(n)
    if noise_type == "traffic":
        if os.path.exists(TRAFFIC_WAV):
            y = load_wav(TRAFFIC_WAV)
            if len(y) < n:
                reps = int(np.ceil(n/len(y)))
                y = np.tile(y, reps)
            y = y[:n]
            y = y / (rms(y) + 1e-12)
            return y.astype(np.float32)
        # fallback if traffic.wav not provided:
        return pink_noise(n)
    raise ValueError("Unknown noise_type")

def generate_synthetic_noisy():
    for f in clean_files:
        sid = os.path.splitext(f)[0]
        y = load_wav(os.path.join(CLEAN_DIR, f))

        for nt in NOISE_TYPES:
            noise = get_noise(nt, len(y))
            for snr in SNR_LEVELS:
                yn = add_noise_snr(y, noise, snr_db=snr)
                out_dir = os.path.join(SYN_DIR, f"noise_{nt}_snr{snr}")
                save_wav(os.path.join(out_dir, f"{sid}.wav"), yn)

generate_synthetic_noisy()
print("Synthetic noisy audio generated in:", SYN_DIR)


## 6) Optional: Reverberation (lightweight)

If you want an **optional** reverb experiment:
- Apply 1–2 RT60 settings (e.g., 0.3s small room, 0.8s large room)

Outputs:
- `project/audio/synthetic/reverb_small/`
- `project/audio/synthetic/reverb_large/`

If you skip this section, that's OK.


In [ ]:
DO_REVERB = False  # set True if you want the optional reverb experiment

def synthetic_rir(sr, rt60=0.3, length_s=1.0):
    n = int(sr*length_s)
    t = np.arange(n) / sr
    decay = np.exp(-6.91 * t / max(rt60, 1e-3))
    rir = np.random.randn(n).astype(np.float32) * decay.astype(np.float32)
    rir[0] += 1.0  # direct impulse
    rir = rir / (np.max(np.abs(rir)) + 1e-12)
    return rir.astype(np.float32)

def apply_reverb(y, sr, rt60):
    rir = synthetic_rir(sr, rt60=rt60, length_s=1.0)
    out = fftconvolve(y, rir, mode="full")[:len(y)]
    out = out / (np.max(np.abs(out)) + 1e-12)
    return out.astype(np.float32)

if DO_REVERB:
    for f in clean_files:
        sid = os.path.splitext(f)[0]
        y = load_wav(os.path.join(CLEAN_DIR, f))
        y_small = apply_reverb(y, SR, rt60=0.3)
        y_large = apply_reverb(y, SR, rt60=0.8)
        save_wav(os.path.join(SYN_DIR, "reverb_small", f"{sid}.wav"), y_small)
        save_wav(os.path.join(SYN_DIR, "reverb_large", f"{sid}.wav"), y_large)
    print("Reverb generated.")
else:
    print("Skipping reverb (set DO_REVERB=True to run).")


## 7) Index the dataset (clean + synthetic + real)

We build a table with columns:
- split: clean / synthetic / real
- condition: clean OR noise_* OR reverb_* OR env1/env2
- sentence_id: s01..s10
- audio_path


In [ ]:
def index_split(split, base_dir, is_flat=False):
    rows = []
    if is_flat:
        for f in list_wavs(base_dir):
            sid = os.path.splitext(f)[0]
            rows.append({"split":split, "condition":split, "sentence_id":sid, "audio_path":os.path.join(base_dir,f)})
        return rows

    if not os.path.exists(base_dir):
        return rows
    for cond in sorted(os.listdir(base_dir)):
        cond_dir = os.path.join(base_dir, cond)
        if not os.path.isdir(cond_dir):
            continue
        for f in list_wavs(cond_dir):
            sid = os.path.splitext(f)[0]
            rows.append({"split":split, "condition":cond, "sentence_id":sid, "audio_path":os.path.join(cond_dir,f)})
    return rows

dataset_rows = []
dataset_rows += index_split("clean", CLEAN_DIR, is_flat=True)
dataset_rows += index_split("synthetic", SYN_DIR, is_flat=False)
dataset_rows += index_split("real", REAL_DIR, is_flat=False)

dataset = pd.DataFrame(dataset_rows)
dataset.head(), dataset["split"].value_counts()


## 8) Configure your ≥ 5 ASR models

Recommended starter set:
- **Whisper** sizes (tiny/base/small/medium/large)
- **Hugging Face** ASR pipelines (wav2vec2 / conformer / etc.)


> Tip: Start with 1 model to test pipeline, then expand to 5.


In [ ]:
# ---- Whisper wrapper ----
def transcribe_whisper(model_size, audio_path):
    import whisper
    model = whisper.load_model(model_size)
    out = model.transcribe(audio_path)
    return (out.get("text","") or "").strip()

# ---- HF wrapper ----
def build_hf_asr(model_id, device=-1):
    from transformers import pipeline
    return pipeline(
        "automatic-speech-recognition",
        model=model_id,
        device=device,
        chunk_length_s=20,
        stride_length_s=(4, 2),
    )

def transcribe_hf(pipe, audio_path):
    out = pipe(audio_path)
    if isinstance(out, dict) and "text" in out:
        return out["text"].strip()
    return str(out).strip()

DEVICE = -1  # set 0 if you have GPU

ASR_MODELS = [
    {"name":"whisper_base", "type":"whisper", "id":"base"},
    {"name":"whisper_small", "type":"whisper", "id":"small"},
    {"name":"wav2vec2_960h", "type":"hf", "id":"facebook/wav2vec2-base-960h"},
    # TODO: add at least 2 more models (hf or whisper)
    # {"name":"...", "type":"hf", "id":"..."},
    # {"name":"...", "type":"hf", "id":"..."},
]

print("Configured models:", [m["name"] for m in ASR_MODELS])


## 9) Run ASR inference

This produces a `results` DataFrame with:
- model, split, condition, sentence_id
- ref (ground truth), hyp (prediction)
- WER/CER/S/I/D


In [ ]:
def normalize_text(s):
    s = (s or "").lower().strip()
    s = re.sub(r"[^a-z0-9' ]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

gt_map = {r.sentence_id: r.text for r in gt.itertuples(index=False)}
gram_map = {r.sentence_id: int(r.is_grammatical) for r in gt.itertuples(index=False)}

def compute_metrics(ref, hyp):
    m = compute_measures(ref, hyp)
    return {
        "WER": m["wer"],
        "CER": cer(ref, hyp),
        "S": m["substitutions"],
        "I": m["insertions"],
        "D": m["deletions"],
        "H": m["hits"],
    }

def run_asr(dataset_df, models):
    # build hf pipelines once
    hf = {}
    for m in models:
        if m["type"] == "hf":
            hf[m["name"]] = build_hf_asr(m["id"], device=DEVICE)

    rows = []
    for m in models:
        for r in dataset_df.itertuples(index=False):
            ref = gt_map[r.sentence_id]
            if m["type"] == "whisper":
                hyp = transcribe_whisper(m["id"], r.audio_path)
            else:
                hyp = transcribe_hf(hf[m["name"]], r.audio_path)

            ref_n = normalize_text(ref)
            hyp_n = normalize_text(hyp)

            met = compute_metrics(ref_n, hyp_n)
            rows.append({
                "model": m["name"],
                "split": r.split,
                "condition": r.condition,
                "sentence_id": r.sentence_id,
                "is_grammatical": gram_map[r.sentence_id],
                "audio_path": r.audio_path,
                "ref": ref,
                "hyp": hyp,
                **met
            })
    return pd.DataFrame(rows)

# Optional: quick test subset
# ds = dataset.query("split=='clean'").copy()
ds = dataset.copy()

# Uncomment to run:
# results = run_asr(ds, ASR_MODELS)
# results.to_csv(os.path.join(MET_DIR, "results_all.csv"), index=False)
# results.head()
print("Ready. Uncomment the run block to execute ASR.")


## 10) Summaries and plots (after you have `results`)

You should produce:
1) WER by condition (per model)
2) Clean vs synthetic vs real comparisons
3) Grammatical vs ungrammatical comparisons


In [ ]:
def summarize(results):
    return (results
            .groupby(["model","split","condition"], as_index=False)
            .agg(WER_mean=("WER","mean"),
                 WER_std=("WER","std"),
                 CER_mean=("CER","mean"),
                 S_sum=("S","sum"),
                 I_sum=("I","sum"),
                 D_sum=("D","sum")))

def plot_wer(summary, title):
    piv = summary.pivot_table(index="condition", columns="model", values="WER_mean")
    ax = piv.plot(kind="bar")
    ax.set_title(title)
    ax.set_ylabel("WER")
    plt.tight_layout()
    plt.show()

# Example:
# summary = summarize(results)
# summary.to_csv(os.path.join(MET_DIR, "summary.csv"), index=False)
# plot_wer(summary[summary.split=="synthetic"], "Synthetic Conditions — WER")


## 11) Grammatical vs Ungrammatical Analysis 

Compute WER for:
- grammatical sentences (7)
- ungrammatical sentences (3)

Then compare: do ungrammatical sentences produce more **substitutions**?


In [ ]:
def grammatical_analysis(results):
    grp = (results
           .groupby(["model","split","condition","is_grammatical"], as_index=False)
           .agg(WER_mean=("WER","mean"),
                S_sum=("S","sum"),
                I_sum=("I","sum"),
                D_sum=("D","sum")))
    return grp

# Example:
# ga = grammatical_analysis(results)
# ga.to_csv(os.path.join(MET_DIR, "grammatical_vs_ungrammatical.csv"), index=False)
# ga.head()


## 12) Error examples (Qualitative analysis)

Pick at least **3-4** interesting failure cases and discuss in your report:
- Was the error due to noise masking?
- Were numbers/named entities misrecognized?
- Did ungrammatical structure confuse decoding?


In [ ]:
def worst_cases(results, n=15):
    cols = ["model","split","condition","sentence_id","is_grammatical","WER","S","I","D","ref","hyp"]
    return results.sort_values("WER", ascending=False)[cols].head(n)

# Example:
# worst_cases(results, n=20)
